# real_groundtruth_multiregion.ipynb

Multi-region version of `real_groundtruth_downscale.ipynb`.

The four-scene protocol is a case study: tens of thousands of HEALPix cells,
but **four patches and one sensor**. This notebook runs the *same seven steps*
on the **40 geographically distributed regions** already used by the synthetic
multi-region validation, so the real-data claim gets the same statistical
treatment as the synthetic one.

**Statistical unit is the region.** One patch per region, ten regions per
scene class. The bootstrap below therefore resamples 10 independent values per
class -- a plain bootstrap over regions, not a cluster bootstrap (with a single
patch per region there is no within-region clustering left to absorb).

**Protocol** (identical to the four-scene notebook, see
`tests/real_groundtruth_common_tools.py`):

1. fetch a real, native 10 m Sentinel-2 B04 patch at the region anchor
2. PSF-aware resample the undegraded data to `NATIVE_LEVEL` (20)
3. plain no-PSF NESTED downgrade to `REF_LEVEL` (19) -- **the reference**
4. Gaussian-blur to the coarse effective resolution (FWHM 25 m)
5. point-sample every `BLOCK`-th pixel (20 m observation)
6. PSF-aware resample the coarse samples onto `REF_LEVEL`
7. compare against the reference, cell-for-cell, at `REF_LEVEL`

**Prerequisite**: `multi_patch_latitude_validation.ipynb` must have been run at
least once -- this notebook reuses the site manifest it writes
(`tables/multi_patch_sites.csv`).

## Setup

In [ ]:
import sys
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start=None, marker="healpix_resample"):
    """Walk upward until a directory containing `marker` is found."""
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the repo root above {p}.")


REPO_ROOT = find_repo_root()
MODULE_PATH = REPO_ROOT / "tests" / "real_groundtruth_common_tools.py"
print("module:", MODULE_PATH, "(exists:", MODULE_PATH.exists(), ")")

spec = importlib.util.spec_from_file_location("real_groundtruth_common_tools", MODULE_PATH)
gt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gt)

gt.describe_config()


## Step 0 -- register the 40 regions

One patch per region, taken at lattice position (1, 1), i.e. the region anchor
itself. The patch is chosen by **position, before any data is fetched or
scored**, so the selection cannot depend on the results.

In [ ]:
scenes = gt.load_region_sites()          # patch_row=patch_col=1 by default
print(f"\n{len(scenes)} scenes registered, e.g.:")
for s in scenes[:3]:
    c = gt.region_coordinates[s]
    print(f"  {s:44s} {c['location']:22s} "
          f"lat={c['wgs84']['lat']:7.3f} lon={c['wgs84']['lon']:8.3f}")


## Step 1 -- fetch the Sentinel-2 patches

**This adds 40 new Sentinel-2 patches to the frozen input bundle.** They are
not in the DOI archive yet, so `OFFLINE` has to be turned off for this one
fetch. Re-publish the archive afterwards and set it back to `True`.

Cloud cover is capped at `gt.REGION_CLOUD_MAX` (10%) over
`gt.REGION_DATE_WINDOW`. Regions with no cloud-free acquisition in that window
simply fail here and are excluded downstream -- a criterion that depends only
on data availability, never on a method score.

In [ ]:
gt.OFFLINE = False        # required: these patches are not in the archive yet
status = gt.fetch_region_sites()
gt.OFFLINE = True         # restore the publication default

display(status[status.status != "ok"])
available = status.loc[status.status == "ok", "scene"].tolist()
print(f"{len(available)} / {len(status)} patches usable.")


## Steps 2-7 -- run the protocol on every region

Same code path as the four-scene notebook: `run_all()` per scene, then the
per-region rows are concatenated with their scene class and region id.

In [ ]:
metrics, failures = gt.run_multiregion(scenes=available, force=False)

print()
print(metrics.groupby("scene_class").region_id.nunique().rename("regions").to_string())
display(failures)


## Region-level aggregation

Per scene class: mean RMSE per method over regions with a percentile bootstrap
interval, and the paired per-region difference against PSF-aware.

In [ ]:
summary, comparisons = gt.summarize_multiregion(metrics)
display(summary)


In [ ]:
display(comparisons[["scene_class", "competitor", "n_regions",
                     "mean_delta_rmse_competitor_minus_matched",
                     "delta_ci95_low", "delta_ci95_high",
                     "matched_win_fraction"]])

print("win fraction 1.0 everywhere:", bool((comparisons.matched_win_fraction == 1).all()))
print("every paired interval excludes zero:", bool((comparisons.delta_ci95_low > 0).all()))


## Paper table

In [ ]:
table = gt.format_multiregion_table(summary, comparisons)
display(table)


## Figure -- per-class RMSE with bootstrap intervals

In [ ]:
methods = ["psf_aware", "classical_nearest", "classical_linear",
           "classical_cubic", "richardson_lucy"]
labels = {"psf_aware": "PSF-aware", "classical_nearest": "Nearest",
          "classical_linear": "Bilinear", "classical_cubic": "Bicubic",
          "richardson_lucy": "Richardson-Lucy"}
classes = sorted(summary.scene_class.unique())

fig, ax = plt.subplots(figsize=(9, 4.5))
width = 0.15
x = np.arange(len(classes))
for i, m in enumerate(methods):
    sub = summary[summary.method == m].set_index("scene_class")
    vals = [sub.loc[c, "mean_rmse"] if c in sub.index else np.nan for c in classes]
    lo = [sub.loc[c, "mean_rmse"] - sub.loc[c, "rmse_ci95_low"] if c in sub.index else 0 for c in classes]
    hi = [sub.loc[c, "rmse_ci95_high"] - sub.loc[c, "mean_rmse"] if c in sub.index else 0 for c in classes]
    ax.bar(x + (i - len(methods) / 2) * width, vals, width,
           yerr=[lo, hi], capsize=2, label=labels[m])
ax.set_xticks(x); ax.set_xticklabels(classes)
ax.set_ylabel(f"RMSE vs. reference (HEALPix level {gt.REF_LEVEL})")
ax.set_title("Real-data downscaling over 40 regions (10 per class)")
ax.legend(fontsize=8)
fig.tight_layout()
gt.FIG_DIR.mkdir(exist_ok=True)
fig.savefig(gt.FIG_DIR / "real_groundtruth_multiregion_rmse.pdf", dpi=200)


## Width-mismatch control (real-data analogue of the +-50% arms)

The degradation stays at 25 m; only the **assumed** reconstruction width
changes. Note the asymmetry that makes this informative: only PSF-aware and
Richardson--Lucy use the assumed width, so a width error costs exactly the two
response-modelling methods and leaves the geometric baselines untouched.

Run on the four benchmark scenes first (cheap); pass `scenes=available` to
repeat it over all 40 regions.

In [ ]:
df_wide   = gt.run_width_mismatch(scale=1.5)   # overestimated response
df_narrow = gt.run_width_mismatch(scale=0.5)   # underestimated response
df_matched = gt.run_all(force=False)

comp = (df_matched[["scene", "method", "rmse"]]
        .merge(df_wide[["scene", "method", "rmse"]], on=["scene", "method"],
               suffixes=("_matched", "_plus50"))
        .merge(df_narrow[["scene", "method", "rmse"]].rename(columns={"rmse": "rmse_minus50"}),
               on=["scene", "method"]))
comp["plus50_pct"] = 100 * (comp.rmse_plus50 - comp.rmse_matched) / comp.rmse_matched
comp["minus50_pct"] = 100 * (comp.rmse_minus50 - comp.rmse_matched) / comp.rmse_matched
display(comp)


## Notes

- Every result-changing setting (`REF_LEVEL`, `BLOCK`, `RECON_FWHM_M`,
  `MAX_ITER`, `REFERENCE_MODE`, `BASELINE_ESTIMAND`, degradation mode) is
  encoded in the cache and CSV filenames via `gt._cache_suffix()`, so variants
  never overwrite each other or the four-scene result the paper's Table VI is
  built from.
- `gt.run_multiregion()` skips a region only when its patch is missing or its
  reference is degenerate. Both criteria are evaluated before any method is
  scored.
- To rerun everything from scratch, pass `force=True`; otherwise cached
  intermediate `.npz` files are reused.